# 5 skyrius. Eksperimentų rezultatai ir jų analizė
## 5.2 poskyris. STFT parametrų ir segmentų ilgio paieškos rezultatai

## 1. Importai ir nustatymai

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import FuncFormatter
import os, warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams.update({
    'font.family':       'DejaVu Sans',
    'font.size':         10,
    'axes.titlesize':    12,
    'axes.labelsize':    11,
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'axes.grid':         False,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.spines.left':  False,
    'axes.spines.bottom':False,
})

CSV_PATH    = os.path.join('results', 'all_runs.csv')
FIGURES_DIR = 'figures'
os.makedirs(FIGURES_DIR, exist_ok=True)
print('Importai sekmingi.')
print(f'Paveikslu katalogas: {os.path.abspath(FIGURES_DIR)}')

---
## 2. Duomenų nuskaitymas ir diagnostika

In [ ]:
df_all = pd.read_csv(CSV_PATH)
print(f'Is viso eiluciu: {len(df_all)}')
print(f'Is viso stulpeliu: {len(df_all.columns)}')
print('\n=== Stulpeliai ===')
for i, c in enumerate(df_all.columns): print(f'  [{i:2d}] {c}')
print('\n=== Kelios pirmos eilutes ===')
display(df_all[['experiment_id','model_name','n_fft','segment_seconds',
               'best_val_loss','mean_pesq_enhanced','mean_stoi_enhanced']].head(4))
print('\n=== Visi unikalus experiment_id ===')
for eid in sorted(df_all['experiment_id'].unique()): print(f'  {eid}')

---
## 3. Stulpelių susiejimas

In [ ]:
COL_MAP = {
    'eksperimento_id'      : 'experiment_id',
    'modelio_pavadinimas'  : 'model_name',
    'stft_lango_dydis'     : 'n_fft',
    'segmento_trukme'      : 'segment_seconds',
    'val_nuostolis'        : 'best_val_loss',
    'pesq_pries'           : 'mean_pesq_noisy',
    'pesq_po'              : 'mean_pesq_enhanced',
    'delta_pesq'           : 'mean_delta_pesq',
    'stoi_pries'           : 'mean_stoi_noisy',
    'stoi_po'              : 'mean_stoi_enhanced',
    'delta_stoi'           : 'mean_delta_stoi',
    'delta_snr'            : 'mean_delta_snr_est',
}
print('=== STULPELIU SUSIEJIMAS ===')
for alias, col in COL_MAP.items():
    ok = col in df_all.columns
    pvz = df_all[col].iloc[0] if ok else 'NERASTAS'
    print(f'  {"OK" if ok else "KLAIDA"}  {alias:28s}  {col}  (pvz.: {pvz})')

---
## 4. Parametrų paieškos eilučių filtravimas

In [ ]:
df_sweep_all = df_all[df_all['experiment_id'].str.startswith('sweep_')].copy()
df_sweep     = df_sweep_all[df_sweep_all['model_name'] == 'UNetDilatedMaskCNN'].copy()
COLS_RODYTI  = ['experiment_id','n_fft','segment_seconds','best_val_loss',
               'mean_pesq_enhanced','mean_stoi_enhanced','mean_delta_snr_est']
print(f'sweep_ eiluciu is viso: {len(df_sweep_all)}')
print(f'UNetDilatedMaskCNN sweep eiluciu: {len(df_sweep)}')
print(f'Unikalus n_fft: {sorted(df_sweep["n_fft"].unique())}')
print(f'Unikalus segment_seconds: {sorted(df_sweep["segment_seconds"].unique())}')
print('\n=== Visos sweep eilutes ===')
display(df_sweep[COLS_RODYTI].reset_index(drop=True))
n_uniq = df_sweep.groupby(['n_fft','segment_seconds']).ngroups
print(f'\nUnikalus (n_fft, segment_seconds) deriniai: {n_uniq}')
if n_uniq == 9:
    print('Pilnas 3x3 eksperimentinis dizainas.')
elif n_uniq < 9:
    print(f'Nerasta 9 deriniu - tik {n_uniq}. OFAT dizainas.')
    print('   Heatmap tures NaN langeliu (rodoma kaip dash).')

---
## 5. Dublikatų tikrinimas

In [ ]:
dup_mask = df_sweep.duplicated(subset=['n_fft','segment_seconds'], keep=False)
df_dups  = df_sweep[dup_mask]
if len(df_dups):
    print(f'Rasta {len(df_dups)} eiluciu su pasikartojanciais (n_fft, seg):')
    display(df_dups[COLS_RODYTI].reset_index(drop=True))
    print('Sprendimas: naudojamas vidurkis.')
else:
    print('Dublikatu nerasta.')

df_uniq = (
    df_sweep
    .groupby(['n_fft','segment_seconds'], as_index=False)
    .agg(
        experiment_id      = ('experiment_id',      lambda x: ' / '.join(sorted(x.unique()))),
        best_val_loss      = ('best_val_loss',      'mean'),
        mean_pesq_noisy    = ('mean_pesq_noisy',    'mean'),
        mean_pesq_enhanced = ('mean_pesq_enhanced', 'mean'),
        mean_delta_pesq    = ('mean_delta_pesq',    'mean'),
        mean_stoi_noisy    = ('mean_stoi_noisy',    'mean'),
        mean_stoi_enhanced = ('mean_stoi_enhanced', 'mean'),
        mean_delta_stoi    = ('mean_delta_stoi',    'mean'),
        mean_delta_snr_est = ('mean_delta_snr_est', 'mean'),
    )
)
print(f'Po vidurkio: {len(df_uniq)} unikalus deriniai')
display(df_uniq[['experiment_id','n_fft','segment_seconds','best_val_loss','mean_pesq_enhanced']].reset_index(drop=True))

---
## 6. Metrikų skaičiavimas ir pivot lentelės

In [ ]:
SEGS  = sorted(df_uniq['segment_seconds'].unique())
NFFTS = sorted(df_uniq['n_fft'].unique())

def make_pivot(col):
    return df_uniq.pivot(index='segment_seconds', columns='n_fft', values=col)\
                  .reindex(index=SEGS, columns=NFFTS)

piv_val  = make_pivot('best_val_loss')
piv_pesq = make_pivot('mean_pesq_enhanced')

PESQ_REF = df_sweep['mean_pesq_noisy'].mean()
STOI_REF = df_sweep['mean_stoi_noisy'].mean()
print(f'PESQ atskaitos reikšme: {PESQ_REF:.4f}')
print(f'STOI atskaitos reikšme: {STOI_REF:.4f}')

best_val_coord  = tuple(np.unravel_index(np.nanargmin(piv_val.values),  piv_val.values.shape))
best_pesq_coord = tuple(np.unravel_index(np.nanargmax(piv_pesq.values), piv_pesq.values.shape))
best_val_seg  = SEGS[best_val_coord[0]];  best_val_nfft  = NFFTS[best_val_coord[1]]
best_pesq_seg = SEGS[best_pesq_coord[0]]; best_pesq_nfft = NFFTS[best_pesq_coord[1]]
print(f'Geriausias val_loss: n_fft={best_val_nfft}, seg={best_val_seg}s  {piv_val.values[best_val_coord]:.5f}')
print(f'Geriausias PESQ:     n_fft={best_pesq_nfft}, seg={best_pesq_seg}s  {piv_pesq.values[best_pesq_coord]:.3f}')
if best_val_coord == best_pesq_coord:
    print('Abu kriterijai rodo ta pati derini.')
else:
    print('Geriausias val_loss ir geriausias PESQ yra skirtingi deriniai.')
print('\n=== Validavimo nuostolio pivot ===')
display(piv_val.rename_axis('seg, s').rename_axis('n_fft', axis=1).round(5))
print('\n=== PESQ pivot ===')
display(piv_pesq.rename_axis('seg, s').rename_axis('n_fft', axis=1).round(3))

---
## 7. Santraukos lentelių išsaugojimas

In [ ]:
san = df_sweep[['experiment_id','n_fft','segment_seconds','best_val_loss',
                'mean_pesq_noisy','mean_pesq_enhanced','mean_delta_pesq',
                'mean_stoi_noisy','mean_stoi_enhanced','mean_delta_stoi',
                'mean_delta_snr_est']].copy()
san = san.sort_values('best_val_loss').reset_index(drop=True)
san.columns = [
    'Eksperimentas','STFT lango dydis','Segmento trukme, s',
    'Geriausias validavimo nuostolis',
    'PESQ pries','PESQ po','DPESQ',
    'STOI pries','STOI po','DSTOI','DSNR',
]
p_san = os.path.join(FIGURES_DIR, 'parametru_paieska_santrauka.csv')
san.to_csv(p_san, index=False, encoding='utf-8-sig')
print(f'OK {p_san}  ({len(san)} eilutes)')
best_row = san.iloc[0:1].copy()
p_best   = os.path.join(FIGURES_DIR, 'parametru_paieska_geriausias.csv')
best_row.to_csv(p_best, index=False, encoding='utf-8-sig')
print(f'OK {p_best}')
display(best_row)

---
## 8. Pagalbinės funkcijos (WCAG kontrastas + lietuviškas kablelis)

In [ ]:
# ── Teksto kontrasto skaiciavimas pagal WCAG 2.1 ─────────────────────────────
# Matematiskai optimali riba: sqrt(1.05 * 0.05) - 0.05 ~= 0.179
LUMI_THRESHOLD = 0.179

def srgb_to_linear(c):
    return c / 12.92 if c <= 0.04045 else ((c + 0.055) / 1.055) ** 2.4

def relative_luminance(rgba):
    r, g, b = [srgb_to_linear(rgba[k]) for k in range(3)]
    return 0.2126*r + 0.7152*g + 0.0722*b

def txt_color_for_bg(cmap_obj, norm, value):
    lum = relative_luminance(cmap_obj(norm(value)))
    return 'white' if lum < LUMI_THRESHOLD else '#1A1A1A'

# ── Bendra heatmap funkcija ───────────────────────────────────────────────────
def draw_heatmap(pivot, title, cbar_label, fmt_fn, cmap_name,
                 best_coord, fig_name, note=None):
    data = pivot.values.astype(float)
    n_rows, n_cols = data.shape
    row_lbl = [f'{int(s)} s' for s in SEGS]
    col_lbl = [str(int(n)) for n in NFFTS]
    vmin, vmax = np.nanmin(data), np.nanmax(data)
    cmap_obj = plt.cm.get_cmap(cmap_name).copy()
    cmap_obj.set_bad('#DEDEDE')
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

    fig_h = 4.9 + (0.35 if note else 0.0)
    fig, ax = plt.subplots(figsize=(7.5, fig_h))
    ax.set_aspect('equal')
    ax.add_patch(plt.Rectangle((-0.5, -0.5), n_cols, n_rows,
                                color='#F8F8F8', zorder=0))
    im = ax.imshow(np.ma.masked_invalid(data), cmap=cmap_obj,
                   norm=norm, aspect='equal', zorder=1)
    for x in np.arange(-0.5, n_cols, 1): ax.axvline(x, color='white', lw=1.4, zorder=2)
    for y in np.arange(-0.5, n_rows, 1): ax.axhline(y, color='white', lw=1.4, zorder=2)

    for i in range(n_rows):
        for j in range(n_cols):
            v = data[i, j]
            if np.isnan(v):
                ax.text(j, i, '-', ha='center', va='center',
                        fontsize=14, color='#AAAAAA', zorder=3)
            else:
                tcol = txt_color_for_bg(cmap_obj, norm, v)
                bold = (i == best_coord[0] and j == best_coord[1])
                ax.text(j, i, fmt_fn(v), ha='center', va='center', fontsize=11,
                        color=tcol, fontweight='bold' if bold else 'normal', zorder=3)

    bi, bj = best_coord
    ax.add_patch(plt.Rectangle((bj-0.5, bi-0.5), 1, 1,
                                fill=False, edgecolor='#1B5E20',
                                linewidth=3.2, zorder=5))
    ax.text(bj+0.44, bi-0.44, chr(9733), ha='right', va='top',
            fontsize=9, color='#1B5E20', zorder=6)

    ax.set_xticks(range(n_cols)); ax.set_xticklabels(col_lbl, fontsize=11)
    ax.set_yticks(range(n_rows)); ax.set_yticklabels(row_lbl, fontsize=11)
    ax.set_xlabel('STFT lango dydis (n_fft)', fontsize=11, labelpad=8)
    ax.set_ylabel('Segmento trukme', fontsize=11, labelpad=8)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=12)
    ax.tick_params(length=0)

    cb = fig.colorbar(im, ax=ax, shrink=0.82, pad=0.02)
    cb.set_label(cbar_label, fontsize=10)
    # Spalvu juostos zymos su lietuvišku kableliu (tas pats fmt_fn)
    cb.ax.yaxis.set_major_formatter(
        FuncFormatter(lambda x, _: fmt_fn(x))
    )
    cb.ax.tick_params(labelsize=9)

    if note:
        fig.text(0.5, 0.01, note, ha='center', va='bottom',
                 fontsize=8.5, color='#555555', style='italic')
    plt.tight_layout(rect=[0, 0.04 if note else 0.0, 1, 1])
    p = os.path.join(FIGURES_DIR, fig_name)
    fig.savefig(p+'.png', dpi=300, bbox_inches='tight', facecolor='white')
    fig.savefig(p+'.svg',           bbox_inches='tight', facecolor='white')
    print(f'  OK {fig_name}')
    plt.show()

print('Pagalbines funkcijos apibreZtos.')

---
## 9. Validavimo nuostolio šilumos žemėlapis

In [ ]:
draw_heatmap(
    pivot      = piv_val,
    title      = 'STFT lango dydžio ir segmento trukmės\nįtaka validavimo nuostoliui',
    cbar_label = 'Validavimo nuostolis (MSE)',
    fmt_fn     = lambda v: f'{v:.5f}'.replace('.', ','),
    cmap_name  = 'YlGn_r',
    best_coord = best_val_coord,
    fig_name   = '22_parametru_paieska_validavimo_nuostolis',
)

---
## 10. PESQ šilumos žemėlapis

In [ ]:
pesq_ref_str = f'{PESQ_REF:.3f}'.replace('.', ',')
draw_heatmap(
    pivot      = piv_pesq,
    title      = 'STFT lango dydžio ir segmento trukmės\nįtaka PESQ rezultatui',
    cbar_label = 'PESQ reikšmė (po apdorojimo)',
    fmt_fn     = lambda v: f'{v:.3f}'.replace('.', ','),
    cmap_name  = 'YlGn',
    best_coord = best_pesq_coord,
    fig_name   = '23_parametru_paieska_pesq',
    note       = f'Atskaitos PESQ (triukšmingas signalas prieš apdorojimą): {pesq_ref_str}',
)

---
## 11. Rezultatų failų patikra

In [ ]:
print('=== REZULTATŲ FAILŲ PATIKRA ===')
print(f'Sweep eiluciu: {len(df_sweep)}')
print(f'Unikalus deriniai: {len(df_uniq)}')
for name in ['22_parametru_paieska_validavimo_nuostolis','23_parametru_paieska_pesq']:
    for ext in ('png','svg'):
        p = os.path.join(FIGURES_DIR, name+'.'+ext)
        ok = os.path.exists(p)
        kb = os.path.getsize(p)//1024 if ok else 0
        print(f'  {"OK" if ok else "KLAIDA"}  {name}.{ext}  ({kb} KB)')
for f in ('parametru_paieska_santrauka.csv','parametru_paieska_geriausias.csv'):
    p = os.path.join(FIGURES_DIR, f)
    print(f'  {"OK" if os.path.exists(p) else "KLAIDA"}  {f}')
print('\n=== Santrauka ===')
display(san)